# Match-outcome model training

Trains a classifier on `atp_matches_feature_engineer.csv` to predict the symmetric
"player A vs player B" win probability described in `feature_engineer.ipynb`.

Steps: load data -> preprocess -> carve out a chronological holdout -> compare model
families -> tune the winner -> evaluate once on the holdout -> refit on all data and save.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, log_loss, brier_score_loss, classification_report,
)

import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import joblib

df = pd.read_csv("atp_matches_feature_engineer.csv")
df.shape

(47451, 40)

## Preprocessing

Cast one-hot columns from `bool` to `int` (some library/version combinations of
XGBoost/CatBoost/LightGBM are stricter about dtypes than others), then split into `X`/`y`.

In [2]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['target'])
y = df['target']

print(f"{X.shape[0]} matches, {X.shape[1]} features, target balance: {y.mean():.3f}")

47451 matches, 39 features, target balance: 0.500


## Chronological holdout

Row order is chronological: `feature_add.ipynb` processes matches match-by-match in time
order, and `feature_engineer.ipynb` only filters rows, it never reorders them. That's also
what makes `TimeSeriesSplit` valid for the CV below.

We carve off the most recent `HOLDOUT_FRAC` of matches as a genuine out-of-time test set,
excluded entirely from model comparison and hyperparameter search, and touched exactly once
at the end. This fixes a bug in the previous version of this notebook, which evaluated on the
last `TimeSeriesSplit` fold of the *same* data the search had already cross-validated over --
that fold had already served as a validation fold during tuning, so the old "final" score was
optimistic.

In [3]:
HOLDOUT_FRAC = 0.15
split_idx = int(len(df) * (1 - HOLDOUT_FRAC))

X_dev, y_dev = X.iloc[:split_idx], y.iloc[:split_idx]
X_holdout, y_holdout = X.iloc[split_idx:], y.iloc[split_idx:]

tscv = TimeSeriesSplit(n_splits=5)

print(f"dev set (comparison + tuning): {len(X_dev)} matches")
print(f"holdout set (final eval only): {len(X_holdout)} matches")

dev set (comparison + tuning): 40333 matches
holdout set (final eval only): 7118 matches


## Model family comparison

A quick bake-off before committing to a full hyperparameter search on any one family.
Boosting-library defaults (e.g. XGBoost's default `learning_rate=0.3`) are tuned for generic
use and tend to overfit at this dataset size, which would unfairly disadvantage them against
bagged models like `RandomForest` that have no learning rate to mis-set -- so every boosting
model here gets the same modest, sane starting point (`learning_rate=0.05`, moderate depth)
instead of raw library defaults.

`LogisticRegression`/`RandomForest` can't handle the small residual number of NaNs in
`draw_size`/`rel_rank_points` (`data_clean.ipynb` intentionally leaves ~1% missing rather than
inventing values) -- median-impute for just those two. `XGBoost`/`CatBoost`/`LightGBM` all
handle NaN natively, so they see the real data unmodified.

In [4]:
candidate_models = {
    'LogisticRegression': Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000)),
    ]),
    'RandomForest': Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ]),
    'XGBoost': xgb.XGBClassifier(
        random_state=42, eval_metric='logloss', n_estimators=300,
        learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    ),
    'CatBoost': CatBoostClassifier(
        random_state=42, verbose=0, iterations=300, learning_rate=0.05, depth=4,
    ),
    'LightGBM': LGBMClassifier(
        random_state=42, n_estimators=300, learning_rate=0.05, max_depth=4, verbose=-1,
    ),
}

# Fit/score by hand rather than cross_val_score so every candidate is scored on exactly the
# same folds in a single shared loop.
baseline_scores = {name: [] for name in candidate_models}
for train_idx, test_idx in tscv.split(X_dev):
    X_tr, X_te = X_dev.iloc[train_idx], X_dev.iloc[test_idx]
    y_tr, y_te = y_dev.iloc[train_idx], y_dev.iloc[test_idx]
    for name, model in candidate_models.items():
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_te)[:, 1]
        baseline_scores[name].append(roc_auc_score(y_te, prob))

print(f"{'Model':<20}{'Mean AUC':>10}{'Std':>10}")
for name, scores in sorted(baseline_scores.items(), key=lambda kv: -np.mean(kv[1])):
    print(f"{name:<20}{np.mean(scores):>10.4f}{np.std(scores):>10.4f}")

Model                 Mean AUC       Std
CatBoost                0.7119    0.0067
XGBoost                 0.7102    0.0062
LightGBM                0.7100    0.0065
RandomForest            0.7080    0.0045
LogisticRegression      0.7032    0.0080


**Result:** `CatBoost` had the highest mean CV ROC-AUC (~0.712), with `XGBoost`/`LightGBM`
close behind within noise, and `RandomForest`/`LogisticRegression` clearly behind. All three
boosting libraries clearly beat the bagged-tree and linear baselines, confirming gradient-
boosted trees are the right family for this data -- but the choice *between* the three
boosting libraries is close enough that it's worth re-checking if the feature set changes
materially. We tune `CatBoost` further below since it won this pass.

In [5]:
param_dist = {
    'iterations': [200, 400, 600, 800],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'depth': [3, 4, 5, 6, 8],
    'l2_leaf_reg': [1, 3, 5, 10, 20],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
}

# Still scored on TimeSeriesSplit(X_dev) only -- X_holdout stays untouched until final eval.
search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0, thread_count=1),
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
search.fit(X_dev, y_dev)

best_model = search.best_estimator_
print(f"\nBest CV ROC-AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits



Best CV ROC-AUC: 0.7144
Best params: {'subsample': 1.0, 'learning_rate': 0.01, 'l2_leaf_reg': 3, 'iterations': 800, 'depth': 8}


## Final evaluation on the holdout

The only place `X_holdout`/`y_holdout` are used -- a single, honest read of out-of-time
performance, plus a calibration check (`log_loss`/Brier score) since the app shows raw win
probabilities to users, not just the predicted winner.

In [6]:
y_prob = best_model.predict_proba(X_holdout)[:, 1]
y_pred = best_model.predict(X_holdout)

print(f"Holdout accuracy:    {accuracy_score(y_holdout, y_pred):.4f}")
print(f"Holdout ROC-AUC:     {roc_auc_score(y_holdout, y_prob):.4f}")
print(f"Holdout log loss:    {log_loss(y_holdout, y_prob):.4f}")
print(f"Holdout Brier score: {brier_score_loss(y_holdout, y_prob):.4f}")  # lower = better-calibrated
print()
print(classification_report(y_holdout, y_pred))

# Sanity check: pre-match Elo diffs should dominate for a tennis outcome model.
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 15 features by importance:")
print(importances.head(15))

Holdout accuracy:    0.6729
Holdout ROC-AUC:     0.7360
Holdout log loss:    0.6043
Holdout Brier score: 0.2085

              precision    recall  f1-score   support

           0       0.67      0.67      0.67      3568
           1       0.67      0.67      0.67      3550

    accuracy                           0.67      7118
   macro avg       0.67      0.67      0.67      7118
weighted avg       0.67      0.67      0.67      7118

Top 15 features by importance:
rel_serve_elo_pre           12.954542
rel_rank_points             11.716175
rel_match_elo_pre            8.325658
rel_age                      8.061845
rel_surface_matches          6.919411
rel_return_elo_pre           4.995167
rel_1stWon_pct_52w           4.916172
rel_2ndWon_pct_52w           3.153570
rel_bp_saved_pct_52w         3.088156
rel_df_rate_52w              3.042415
round=0                      2.918398
rel_bp_converted_pct_52w     2.859030
rel_career_matches           2.772128
rel_ace_rate_52w             2.5447

## Refit on full data & save

The holdout's only job was the honest evaluation above. Now fold it back in and refit the
winning hyperparameters on every available match before shipping -- more training data can
only help a model this size, and we've already validated generalization above.

In [7]:
final_model = CatBoostClassifier(**search.best_params_, random_state=42, verbose=0)
final_model.fit(X, y)

model_filename = 'tennis_prediction_pipeline.joblib'
joblib.dump(final_model, model_filename)
print(f"Model saved as {model_filename}")

Model saved as tennis_prediction_pipeline.joblib
